# Task 8 — Traffic Sign Detection (GTSDB)
**Stage 1: environment setup and dataset audit**

This notebook is the reproducible Colab entry point. It downloads the selected GTSDB Kaggle dataset, stores it in Google Drive, and audits its structure and annotation format before any label conversion or model training. Do not proceed to training until the dataset audit has been reviewed.

## 1. Mount Google Drive and install tools

The raw dataset and training artifacts are stored in Drive rather than committed to Git. A free Colab GPU can be enabled under Runtime → Change runtime type.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/traffic-sign-detection')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

%pip -q install kaggle ultralytics opencv-python-headless pyyaml pandas pillow matplotlib

## 2. Configure Kaggle access securely

Create/download your Kaggle API token from your Kaggle account settings. Upload the `kaggle.json` file when prompted. Never paste the token into a notebook cell, GitHub, or this chat.

In [ ]:
import os
import shutil
from google.colab import files

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / 'kaggle.json'

if not token_path.exists():
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Upload the Kaggle API token file named kaggle.json.')
    shutil.move('kaggle.json', token_path)
    token_path.chmod(0o600)

print('Kaggle credentials configured locally for this Colab runtime.')

## 3. Download the GTSDB Kaggle dataset

Selected Kaggle dataset: [German Traffic Sign Detection Benchmark — GTSDB](https://www.kaggle.com/datasets/safabouguezzi/german-traffic-sign-detection-benchmark-gtsdb).

The Kaggle mirror must be audited against the original benchmark documentation. In particular, confirm that it contains full road scenes and bounding-box annotations, not only cropped sign images (GTSRB classification data).

In [ ]:
import subprocess
import sys

KAGGLE_DATASET = 'safabouguezzi/german-traffic-sign-detection-benchmark-gtsdb'
download = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(RAW_DIR), '--unzip'],
    text=True, capture_output=True
)
print(download.stdout)
if download.returncode != 0:
    print(download.stderr)
    raise RuntimeError('Kaggle download failed. Check Kaggle access and dataset availability.')
print(f'Dataset downloaded to: {RAW_DIR}')

## 4. Audit files and annotation samples

Run this cell and retain its output. We need the exact directory layout, annotation filenames, column counts, class-ID range, image dimensions, and class mapping before implementing the final converter.

In [ ]:
from collections import Counter
import csv
import itertools

all_files = [p for p in RAW_DIR.rglob('*') if p.is_file()]
print(f'Total files: {len(all_files)}')
for suffix in ('.zip', '.ppm', '.png', '.jpg', '.jpeg', '.csv', '.txt'):
    matches = [p for p in all_files if p.suffix.lower() == suffix]
    print(f'{suffix}: {len(matches)}')
    for p in matches[:8]:
        print('  ', p.relative_to(RAW_DIR))

annotation_candidates = [p for p in all_files if p.suffix.lower() in {'.csv', '.txt'} and any(k in p.name.lower() for k in ('gt', 'ground', 'annot'))]
print('\\nAnnotation candidates:')
for path in annotation_candidates[:10]:
    print('\\nFILE:', path.relative_to(RAW_DIR))
    with path.open('r', encoding='utf-8-sig', errors='replace') as stream:
        for line in itertools.islice(stream, 8):
            print(line.rstrip()[:300])

print('\\nImage examples:')
image_files = [p for p in all_files if p.suffix.lower() in {'.ppm', '.png', '.jpg', '.jpeg'}]
for p in image_files[:5]:
    print(p.relative_to(RAW_DIR))

## 5. Stop and review

**Do not train yet.** Share the output of the audit cell so we can verify the Kaggle mirror's provenance, annotation structure, class-ID mapping, and whether it includes a reliable held-out evaluation partition. We will then implement the exact conversion and leakage-safe split.